### Welcome to Week 6 Day 3!

Let's experiment with a bunch more MCP Servers

In [8]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

True

### The first type of MCP Server: runs locally, everything local

Here's a really interesting one: a knowledge-graph based memory.

It's a persistent memory store of entities, observations about them, and relationships between them.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [3]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', title='Create new entities with observations', description='Create new entities with observations', inputSchema={'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'maxLength': 256}, 'entityType': {'type': 'string', 'maxLength': 256}, 'observations': {'type': 'array', 'items': {'type': 'string', 'maxLength': 4096}, 'maxItems': 100}}, 'required': ['name', 'entityType', 'observations']}, 'maxItems': 50}}, 'required': ['entities'], '$schema': 'http://json-schema.org/draft-07/schema#'}, outputSchema=None, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=False, destructiveHint=None, idempotentHint=True, openWorldHint=None), meta=None),
 Tool(name='search_nodes', title='Search for entities and their relations using text search with relevance ranking', description='Search for entities and their relations using text search with relevance ranking', inputSchema={'type':

In [4]:
#import os
from openai import AsyncOpenAI
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

# ---------------------------------------------------------
# Your custom LLMs converted from LangGraph ChatOpenAI
# to OpenAI Agents SDK compatible models
# ---------------------------------------------------------

adesso_model = OpenAIChatCompletionsModel(
    model="gpt-oss-120b-sovereign",
    openai_client=AsyncOpenAI(
        base_url=os.getenv("ADESSO_BASE_URL"),
        api_key=os.getenv("ADESSO_SOVEREIGN_AI_HUB_KEY"),
    ),
)

adesso_lite_model = OpenAIChatCompletionsModel(
    model="qwen-3.6-35b-sovereign",
    openai_client=AsyncOpenAI(
        base_url=os.getenv("ADESSO_BASE_URL"),
        api_key=os.getenv("ADESSO_SOVEREIGN_AI_HUB_KEY"),
    ),
)

adesso_premium_model = OpenAIChatCompletionsModel(
    model="claude-haiku-4-5",
    openai_client=AsyncOpenAI(
        base_url=os.getenv("ADESSO_BASE_URL"),
        api_key=os.getenv("ADESSO_API_KEY"),
    ),
)

vultr_model = OpenAIChatCompletionsModel(
    model="nvidia/DeepSeek-V3.2-NVFP4",
    openai_client=AsyncOpenAI(
        base_url=os.getenv("VULTR_BASE_URL"),
        api_key=os.getenv("VULTR_API_KEY"),
    ),
)

vultr_premium_model = OpenAIChatCompletionsModel(
    model="zai-org/GLM-5.1-FP8",
    openai_client=AsyncOpenAI(
        base_url=os.getenv("VULTR_BASE_URL"),
        api_key=os.getenv("VULTR_API_KEY"),
    ),
)

cerebras_model = OpenAIChatCompletionsModel(
    model="zai-glm-4.7",
    openai_client=AsyncOpenAI(
        base_url=os.getenv("CEREBRAS_BASE_URL"),
        api_key=os.getenv("CEREBRAS_API_KEY"),
    ),
)

groq_model = OpenAIChatCompletionsModel(
    model="llama-3.1-8b-instant",
    openai_client=AsyncOpenAI(
        base_url=os.getenv("GROQ_BASE_URL"),
        api_key=os.getenv("GROQ_API_KEY"),
    ),
)

In [5]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = adesso_lite_model

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export




Hello Ed! It's great to meet an LLM engineer who's teaching about AI Agents and the MCP protocol. MCP (Model Context Protocol) is indeed fascinating—being able to connect agents with external tools and resources in a standardized way opens up a lot of possibilities.

How can I assist you today? Are you looking for information about MCP, help with course materials, or something else entirely?

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


In [7]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))



Hello, Ed! Based on my memory, I know that you are an LLM engineer and that you teach a course about AI Agents. You're also interested in the MCP protocol, which is a protocol for connecting agents with tools, resources, and prompt templates.

### Check the trace:

https://platform.openai.com/traces

### The 2nd type of MCP server - runs locally, calls a web service

### Brave Search - apologies - this will need another API key! But it's free again.

https://brave.com/search/api/

Set up your account, and put your key in the .env under `BRAVE_API_KEY`

In [9]:
env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}
params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='brave_web_search', title=None, description='Performs a web search using the Brave Search API, ideal for general queries, news, articles, and online content. Use this for broad information gathering, recent events, or when you need diverse web sources. Supports pagination, content filtering, and freshness controls. Maximum 20 results per request, with offset for pagination. ', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query (max 400 chars, 50 words)'}, 'count': {'type': 'number', 'description': 'Number of results (1-20, default 10)', 'default': 10}, 'offset': {'type': 'number', 'description': 'Pagination offset (max 9, default 0)', 'default': 0}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=None),
 Tool(name='brave_local_search', title=None, description="Searches for local businesses and places using Brave's Local Search API. Best for queries related to physical locations, businesses, re

In [10]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = vultr_model

In [11]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Based on my research, here's a summary of the latest news on Amazon's stock price and outlook as of late May 2026:

## **Current Stock Status**
- **Current price**: Around $266-$268 per share (as of late May 2026)
- **Performance**: The stock fell approximately 0.8% on May 22, 2026, from $268.46 to $266.32

## **Recent Financial Performance**
1. **Strong Q1 2026 Results**:
   - Net sales increased 17% year-over-year to $181.5 billion
   - Earnings per share of $2.78, beating expectations by 69.5%
   - AWS (Amazon Web Services) grew 28% year-over-year to $37.6 billion, marking its strongest growth in years
   - Operating income reached $23.9 billion

2. **Q2 2026 Guidance**:
   - Net sales forecast: $194-$199 billion (16-19% year-over-year growth)
   - Operating income guidance: $20-$24 billion

## **Analyst Outlook & Price Targets**
- **Consensus rating**: "Strong Buy" or "Buy" across most analysts
- **41 analysts surveyed** maintain a Buy consensus rating
- **Current average price target**: $312.66 (approximately 17% upside from current levels)
- **High price target**: $370 (Benchmark, April 30, 2026) - about 39% upside
- **Low price target**: $175 (DA Davidson, February 2026)

## **Key Drivers & Outlook**

**Positive Factors**:
1. **AWS Acceleration**: 28% growth is the segment's strongest performance in over three years, driven by AI and cloud investments
2. **AI Investment Payoff**: Amazon's $43.2 billion Q1 capital expenditure focused on AWS and generative AI
3. **Strong Revenue Guidance**: Double-digit growth projections for Q2 2026
4. **Analyst Upgrades**: New Street Research raised price target from $280 to $350 following Q1 earnings
5. **Market Leadership**: Continued dominance in e-commerce and cloud computing

**Risk Factors**:
1. **High Valuation**: Trailing P/E ratio around 31.5, which some consider elevated
2. **Market Volatility**: Some technical indicators showing short-term bearish signals
3. **Investment Pressure**: Continued heavy capital expenditure requirements for AI infrastructure

## **Summary**
Amazon appears to be experiencing strong momentum in 2026, particularly in its AWS cloud business which is seeing its fastest growth in years. The company's heavy investments in AI and cloud infrastructure seem to be paying off with above-expectations performance. Analysts remain overwhelmingly bullish with an average price target suggesting approximately 17% upside potential from current levels. However, the stock's premium valuation and continued need for substantial capital investment remain considerations for investors.

### As usual, check out the trace:

https://platform.openai.com/traces

## And now the third type: running remotely

It's actually really hard to find a "remote MCP server" aka "hosted MCP server" aka "managed MCP server".

It's not a common model for using or sharing MCP servers, and there isn't a standard way to discover remote MCP servers.

Anthropic lists some remote MCP servers, but these are for paid applications with business users:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare has tooling for you to create and deploy your own remote MCP servers, but this does not seem to be a common practice:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# And back to the 2nd type: the Polygon.io MCP Server

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">PLEASE READ!!-</h2>
            <span style="color:#ff7800;">This service for financial market data has both a FREE plan and a PAID plan, and we can use either depending on your appetite.
            </span>
        </td>
    </tr>
</table>

## NEW SECTION: Introducing polygon.io

Polygon.io is a hugely popular financial data provider. It has a free plan and a paid plan. And it also has an MCP Server!

First, read up on polygon.io on their excellent website, including looking at their pricing:

https://polygon.io

### Polygon.io Part 1: Polygon.io free service (the paid will be totally optional, of course!)

1. Please sign up for polygon.io (top right)  
2. Once signed in, please select "Keys" in the left hand navigation
3. Press the blue "New Key" button
4. Copy the key name
5. Edit your .env file and add the row:

`POLYGON_API_KEY=xxxx`

In [ ]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [ ]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

### Wrapped into a python module that caches end of day prices

I've made a python module `market.py` that uses this API to look up share prices.

But the free API is quite heavily rate limited - so I've been a bit sneaky; when you ask for a share price, this function retrieves the entire end-of-day equity market, and caches it in our database.


In [ ]:
from market import get_share_price
get_share_price("AAPL")

In [ ]:
# no rate limiting concerns!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

### And I've made this into an MCP Server

Just as we did with accounts.py; see `market_server.py`

In [ ]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

### Let's try it out!

Hopefully gpt-4o-mini is smart enough to know that the symbol for Apple is AAPL

In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Polygon.io Part 2: Paid Plan - Totally Optional!

If you are interested, you can subscribe to the monthly plan to get more up to date market data, and unlimited API calls.

If you do wish to do this, then it also makes sense to use the full MCP server that Polygon.io has released, to take advantage of all their functionality.



In [ ]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


### Wow that's a lot of tools!

Let's try them out - hopefully the sheer number of tools doesn't overwhelm gpt-4o-mini!

With the $29 monthly plan, we don't have access to some of the APIs, so I've needed to specify which APIs can be called.

If you've splashed out on a bigger plan, feel free to remove my extra constraint..

In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple? Use your get_snapshot_ticker tool to get the latest price."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Setting up your .env file

If you do decide to have a paid plan, please add this to your .env file to indicate:

`POLYGON_PLAN=paid`

And if you decide to go all the way for the realtime API, then please do:

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("You've chosen to subscribe to the paid Polygon plan, so the code will look at prices on a 15 min delay")
elif is_realtime_polygon:
    print("Wowzer - you've chosen to subscribe to the realtime Polygon plan, so the code will look at realtime prices")
else:
    print("According to your .env file, you've chosen to subscribe to the free Polygon plan, so the code will look at EOD prices")

## And that's it for today!

I've removed the part of this lab that uses the "Financial Datasets" mcp server, because it's inferior - more expensive with fewer APIs.

And this way we get to use the same provider for Free and Paid APIs.

But if you want to see the code, just look in the git history for a prior version.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Explore MCP server marketplaces and integrate your own, using all 3 approaches.
            </span>
        </td>
    </tr>
</table>